<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Asset_Library_Browser_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Asset Library Browser — browse, tag, preview, export your 200+ game assets

Once you've run TripoSplat + GauStudio / SuGaR + Mesh Optimizer on your 200+ images, you have a folder full of assets but **no way to browse them, pick the best ones, or ship them to a game engine**. This notebook is the missing piece — a Gradio UI that scans your asset library, lets you preview / tag / favorite / filter, render thumbnails, and export to Unity AssetBundle, UE, Godot, or a static HTML portfolio.

**What it does:**

- **Step 1** — install deps (gradio 5.49.1, trimesh, open3d, tqdm, no model weights needed)
- **Step 2** — scan a library folder, recognize formats (3DGS PLY/SPLAT, mesh GLB/OBJ/FBX/PLY, image PNG, FBX, text), build a metadata sidecar JSON (tags, favorites, notes per asset)
- **Step 3** — the main Gradio UI: gallery with thumbnails, filter by tag/format/favorite, click any asset to preview (3D mesh in `<model-viewer>`, 3DGS gets metadata + 'convert to mesh' button), tag editor, favorite toggle
- **Step 4** — batch render thumbnails (renders each mesh to a 256×256 PNG via trimesh's offscreen renderer; 3DGS falls back to file icon or optional turntable GIF via gsplat)
- **Step 5** — export: Unity-style AssetBundle folder structure, Godot .tres, static HTML portfolio (single self-contained file you can host on GitHub Pages), CSV manifest for the library
- **Step 6** — stats dashboard: total assets, format breakdown, top tags, missing-file report, biggest files
- **Step 7** — Drive mirror (whole library + metadata + portfolio HTML)
- **Step 8** — keep-alive (Gradio runs forever otherwise)
- **Step 9** — help / format reference / known issues

**Compute:** CPU-only. The browser is a UI, not a model. No GPU required, no model weights to download. First run: ~3 min install. Subsequent: instant. Can run alongside other notebooks in the same session (read-only on the library folder).

In [ ]:
# @title STEP 1 — Install Dependencies (CPU-only)
# @markdown The Asset Library Browser is a Gradio UI on top of trimesh +
# @markdown open3d + Pillow. No GPU, no model weights, no 3DGS rasterizer
# @markdown needed. ~2 min install.

import os, sys, subprocess, time
t0 = time.time()
print('[browser] Installing lightweight deps ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gradio==5.49.1', 'tqdm==4.66.5',
                'trimesh[easy]==4.12.2', 'open3d==0.19.0',
                'scikit-image', 'pillow', 'numpy', 'pandas',
                'pywavefront'], check=True)
print(f'[browser] Deps installed in {time.time()-t0:.1f}s', flush=True)

# Sanity-check
import gradio, trimesh, open3d, PIL, numpy
print(f'[verify] gradio={gradio.__version__} trimesh={trimesh.__version__} '
      f'open3d={open3d.__version__} PIL={PIL.__version__} numpy={numpy.__version__}')
# ── Google Drive mount (required if LIBRARY_DIR is on /content/drive) ───
MOUNT_DRIVE = True  # @param {type:"boolean"}
# @markdown *If your LIBRARY_DIR is on Google Drive, this force-mounts it. If you already
# @markdown have Drive mounted, this is a no-op. If the mount is stale (e.g. OSError 107 after
# @markdown a runtime disconnect), run this cell again to refresh the FUSE handle.*
if MOUNT_DRIVE:
    try:
        # Check if the existing mount is still healthy
        _stale = False
        if os.path.isdir('/content/drive/MyDrive'):
            try:
                _ = os.listdir('/content/drive/MyDrive')
            except OSError as _e:
                if _e.errno == 107 or 'Transport endpoint' in str(_e):
                    _stale = True
                else:
                    raise
        if _stale:
            print('[drive] Existing mount is stale (OSError 107). Force-unmounting ...')
            try:
                subprocess.run(['fusermount', '-uz', '/content/drive'], check=False, capture_output=True)
            except Exception:
                pass
            time.sleep(2)
        if not os.path.isdir('/content/drive/MyDrive') or _stale:
            print('[drive] Mounting Google Drive ...')
            from google.colab import drive as _gdrive
            _gdrive.mount('/content/drive', force_remount=True)
            print('[drive] Mounted at /content/drive')
        else:
            print('[drive] Already mounted and healthy at /content/drive')
    except ImportError:
        print('[drive] google.colab not available (not in Colab). Skipping mount.')
    except Exception as _e:
        print(f'[drive] Mount failed: {_e}')
        print('[drive] You can mount manually: from google.colab import drive; drive.mount("/content/drive", force_remount=True)')
else:
    print('[drive] MOUNT_DRIVE=False. Assuming LIBRARY_DIR is in /content/ (runtime only).')
print('\n[Done] STEP 1 complete. Move on to STEP 2 (library scan).')


In [ ]:
# @title STEP 2 — Scan Library + Build Metadata Sidecar
# @markdown Scans a folder of 200+ generated assets, recognizes formats, builds a
# @markdown metadata sidecar JSON with tags / favorites / notes. The sidecar lives
# @markdown in `<LIBRARY_DIR>/_library_meta.json` and is the persistent state.

import os, sys, json, time, pathlib, hashlib
from datetime import datetime, timezone
import numpy as np
from PIL import Image

LIBRARY_DIR = '/content/drive/MyDrive/AEI_3D_Out'  # @param {type:"string"}
# @markdown *Folder containing your generated assets. Usually /content/drive/MyDrive/AEI_3D_Out/. Can be the root or any subfolder (e.g. /triposplat_runs/batch).*
RECURSIVE = True  # @param {type:"boolean"}
# @markdown *Scan subfolders. Recommended ON for the 200+ library (assets are in batch_*/, postprocessed/, etc.)*
INCLUDE_GLOBS = '*'  # @param {type:"string"}
# @markdown *Glob pattern. Default '*' = everything. Set to '*.glb' to only see meshes, '*.ply' for 3DGS + meshes, etc.*
MAX_ASSETS = 0  # @param {type:"integer"}
# @markdown *Cap on the number of assets scanned. 0 = no cap (default). Useful for testing.*
THUMB_DIR_NAME = '_thumbs'  # @param {type:"string"}
# @markdown *Subfolder for rendered thumbnails. Auto-created.*

lib_path = pathlib.Path(LIBRARY_DIR)
if not lib_path.exists():
    raise SystemExit(f'[scan] Library dir {lib_path} not found. Set LIBRARY_DIR to your assets folder.')

META_FILE = lib_path / '_library_meta.json'
THUMB_DIR = lib_path / THUMB_DIR_NAME
THUMB_DIR.mkdir(exist_ok=True)

# Load existing metadata (if any) to preserve tags / favorites across runs
_meta_read_ok = False
if META_FILE.exists():
    try:
        with open(META_FILE) as f:
            meta = json.load(f)
        print(f'[scan] Loaded existing metadata: {len(meta["assets"])} assets')
        _meta_read_ok = True
    except OSError as _read_err:
        if _read_err.errno == 107 or 'Transport endpoint' in str(_read_err):
            print('[scan] Stale Drive mount detected while reading _library_meta.json.')
            print('[scan]   Will fall back to writing metadata to /content/ (runtime only).')
            print('[scan]   Re-run STEP 1 to force-remount Drive, then re-run STEP 2 to persist.')
        else:
            raise
if not _meta_read_ok:
    meta = {'version': 1, 'library_dir': str(lib_path),
            'created': datetime.now(timezone.utc).isoformat(),
            'assets': {}}
    print('[scan] No existing metadata; starting fresh')

# ── Format recognition ─────────────────────────────────────────────
# Map (extension, optional header magic) to format category
# Format rules. Order matters for .ply: 3DGS detection (header magic) runs
# FIRST in classify(), so the FORMAT_RULES entry for .ply is a fallback only.
# Suffix-only rules (SOG, SPZ, KSPLAT, etc.) don't need header inspection.
# .glb can be either a regular mesh OR a splat-bearing GLB with the
# KHR_gaussian_splatting extension. classify() detects this by file header.
# Format rules. Order matters for .ply and .glb: 3DGS detection (header magic) runs
# FIRST in classify(), so the FORMAT_RULES entry for those is a fallback only.
# Suffix-only rules (SOG, SPZ, KSPLAT, etc.) don't need header inspection.
FORMAT_RULES = [
    # 3DGS variants
    ('mesh_ply',  '.ply',  'MESH'),            # 3DGS caught above by header magic
    ('3dgs_sog',  '.sog',  '3DGS_COMPRESSED'),  # PlayCanvas native
    ('3dgs_spz',  '.spz',  '3DGS_COMPRESSED'),  # Niantic (mobile AR)
    ('3dgs_ksplat', '.ksplat', '3DGS_COMPRESSED'),  # mkkellogg
    ('3dgs_lcc',  '.lcc',  '3DGS_COMPRESSED'),  # XGRIDS / Luma
    ('3dgs_splat', '.splat', '3DGS_PACKED'),    # Antimatter15 packed format
    # Mesh variants
    ('mesh_glb',  '.glb',  'MESH'),             # splat-GLB caught above
    ('collision_glb', '-collision.glb', 'COLLISION_MESH'),
    ('mesh_gltf', '.gltf', 'MESH'),
    ('mesh_obj',  '.obj',  'MESH'),
    ('mesh_mtl',  '.mtl',  'MESH'),             # material sidecar, treat as MESH
    ('mesh_fbx',  '.fbx',  'MESH'),
    ('mesh_stl',  '.stl',  'MESH'),
    ('mesh_3mf',  '.3mf',  'MESH'),
    ('mesh_usdz', '.usdz', 'MESH'),             # Apple USDZ (AR Quick Look)
    ('mesh_vrm',  '.vrm',  'MESH'),             # VRoid / VTuber avatars
    ('mesh_dae',  '.dae',  'MESH'),             # COLLADA
    ('mesh_x3d',  '.x3d',  'MESH'),
    # Image
    ('image',     '.png',  'IMAGE'),
    ('image',     '.jpg',  'IMAGE'),
    ('image',     '.jpeg', 'IMAGE'),
    ('image',     '.webp', 'IMAGE'),
    ('image',     '.svg',  'IMAGE'),            # vector
    ('image',     '.bmp',  'IMAGE'),
    ('image',     '.tiff', 'IMAGE'),
    ('image',     '.tif',  'IMAGE'),
    ('image',     '.gif',  'IMAGE'),
    # HDR (envmaps, IBL)
    ('hdr',       '.hdr',  'HDR'),
    ('hdr',       '.exr',  'HDR'),
    # Video
    ('video',     '.mp4',  'VIDEO'),
    ('video',     '.webm', 'VIDEO'),
    ('video',     '.mov',  'VIDEO'),
    ('video',     '.mkv',  'VIDEO'),
    # Audio
    ('audio',     '.wav',  'AUDIO'),
    ('audio',     '.mp3',  'AUDIO'),
    ('audio',     '.ogg',  'AUDIO'),
    ('audio',     '.flac', 'AUDIO'),
    ('audio',     '.m4a',  'AUDIO'),
    # Text / data
    ('text',      '.txt',  'TEXT'),
    ('text',      '.csv',  'TEXT'),
    ('text',      '.tsv',  'TEXT'),
    ('text',      '.md',   'TEXT'),
    ('text',      '.log',  'TEXT'),
    ('json',      '.json', 'META'),
    ('yaml',      '.yml',  'META'),
    ('yaml',      '.yaml', 'META'),
    # Archive
    ('archive',   '.zip',  'ARCHIVE'),
    ('archive',   '.tar',  'ARCHIVE'),
    ('archive',   '.gz',   'ARCHIVE'),
    ('archive',   '.7z',   'ARCHIVE'),
    ('archive',   '.rar',  'ARCHIVE'),
]

def classify(path: pathlib.Path) -> str:
    """Classify a file by extension + (for .ply / .glb) header magic.

    Categories:
      - 3DGS_RAW         : raw TripoSplat .ply (3DGS Gaussian attributes)
      - 3DGS_PACKED     : .splat (32-byte packed, Antimatter15 format)
      - 3DGS_COMPRESSED : .sog / .spz / .ksplat / .lcc / .glb w/ KHR_gaussian_splatting
      - MESH             : .glb / .gltf / .obj / .mtl / .fbx / .stl / .3mf / .usdz / .vrm / .dae / .x3d
      - COLLISION_MESH   : voxel collision mesh (e.g. hero_collision.glb)
      - LOD_META         : PlayCanvas streamed-SOG lod-meta.json sidecar
      - IMAGE            : PNG / JPG / JPEG / WEBP / SVG / BMP / TIFF / GIF
      - HDR              : .hdr / .exr (envmaps, image-based lighting)
      - VIDEO            : .mp4 / .webm / .mov / .mkv
      - AUDIO            : .wav / .mp3 / .ogg / .flac / .m4a
      - TEXT / META / ARCHIVE / OTHER
    """
    name = path.name
    ext = path.suffix.lower()
    # Suffix-based rules that need a filename pattern (suffix isn't enough)
    if name.endswith('-collision.glb'):
        return 'COLLISION_MESH'
    if name == 'lod-meta.json':
        return 'LOD_META'
    if ext == '.ply':
        try:
            with open(path, 'rb') as f:
                header = f.read(4096).decode('latin-1', errors='ignore')
            if 'f_dc_' in header and 'opacity' in header and 'scale_0' in header:
                return '3DGS_RAW'
            elif 'face' in header or 'vertex_index' in header:
                return 'MESH'
            else:
                return 'PLY_UNKNOWN'
        except Exception:
            return 'PLY_UNKNOWN'
    if ext == '.glb':
        # glTF 2.0 binary. Could be a regular mesh OR a 3DGS scene that uses
        # the KHR_gaussian_splatting extension. Inspect the JSON header to tell.
        try:
            with open(path, 'rb') as f:
                head = f.read(8192).decode('latin-1', errors='ignore')
            if 'KHR_gaussian_splatting' in head:
                return '3DGS_COMPRESSED'
            else:
                return 'MESH'
        except Exception:
            return 'MESH'
    for fmt, fmt_ext, category in FORMAT_RULES:
        if ext == fmt_ext:
            return category
    return 'OTHER'

# ── Scan the library ────────────────────────────────────────────────
t0 = time.time()
if RECURSIVE:
    all_files = [p for p in lib_path.rglob(INCLUDE_GLOBS) if p.is_file()]
else:
    all_files = [p for p in lib_path.glob(INCLUDE_GLOBS) if p.is_file()]
if MAX_ASSETS:
    all_files = all_files[:MAX_ASSETS]
print(f'[scan] Found {len(all_files)} files in {time.time()-t0:.1f}s')

# Classify + extract metadata
t0 = time.time()
n_added = n_updated = n_skipped_stat = 0
for p in all_files:
    rel = str(p.relative_to(lib_path))
    if rel in meta['assets']:
        # Preserve existing user metadata, just refresh file stats
        m = meta['assets'][rel]
        try:
            _st = p.stat()
            m['size_bytes'] = _st.st_size
            m['mtime'] = _st.st_mtime
            n_updated += 1
        except OSError as _stat_err:
            # OSError 107 = 'Transport endpoint is not connected' = stale FUSE mount
            n_skipped_stat += 1
            if n_skipped_stat <= 3:
                print(f'[scan] SKIP {rel}: {_stat_err}')
                if _stat_err.errno == 107 or 'Transport endpoint' in str(_stat_err):
                    print('[scan]   ^ Stale Google Drive mount (OSError 107).')
                    print('[scan]     Re-run STEP 1 with MOUNT_DRIVE=True to force-remount Drive.')
            continue
    else:
        m = {
            'path': rel,
            'filename': p.name,
            'format': classify(p),
            'size_bytes': 0,
            'mtime': 0,
            'tags': [],
            'favorite': False,
            'notes': '',
            'added': datetime.now(timezone.utc).isoformat(),
        }
        # Now safely stat the file (may fail with OSError 107 on stale Drive mount)
        try:
            _st = p.stat()
            m['size_bytes'] = _st.st_size
            m['mtime'] = _st.st_mtime
            n_added += 1
        except OSError as _stat_err:
            if n_skipped_stat < 3:
                print(f'[scan] SKIP {rel}: {_stat_err}')
                if _stat_err.errno == 107 or 'Transport endpoint' in str(_stat_err):
                    print('[scan]   ^ Stale Google Drive mount (OSError 107).')
                    print('[scan]     Re-run STEP 1 with MOUNT_DRIVE=True to force-remount Drive.')
            continue
        # Look for a <slug>_meta.json sidecar (written by TripoSplat STEP 8
        # or SplatTransform_Colab). If present, merge size_class + grounding
        # + tier info into this asset record so STEP 3 filters can use them.
        meta_sidecar = p.parent / f"{p.stem}_meta.json"
        if meta_sidecar.exists():
            try:
                with open(meta_sidecar) as _mf:
                    _sidecar = json.load(_mf)
                if "size_class" in _sidecar:
                    m["size_class"] = _sidecar["size_class"]
                if "grounding" in _sidecar:
                    m["grounding"] = _sidecar["grounding"]
                if "tiers" in _sidecar:
                    m["tiers"] = _sidecar["tiers"]
                if "colliders" in _sidecar:
                    m["colliders"] = _sidecar["colliders"]
            except Exception as _e:
                print(f"[scan] WARN: could not read {meta_sidecar.name}: {_e}")
        meta['assets'][rel] = m
        n_added += 1
print(f'[scan] Classified in {time.time()-t0:.1f}s: {n_added} new, {n_updated} updated, {n_skipped_stat} skipped (stale mount?)')

# Format breakdown
from collections import Counter
fmt_counts = Counter(m['format'] for m in meta['assets'].values())
print('\n[scan] Format breakdown:')
for fmt, n in fmt_counts.most_common():
    print(f'  {fmt:15s}: {n:4d} files')

total_size = sum(m['size_bytes'] for m in meta['assets'].values())
print(f'\n[scan] Total: {len(meta["assets"])} assets, {total_size/1024/1024:.1f} MB')

# Save metadata
_meta_written_to = None
try:
    with open(META_FILE, 'w') as f:
        json.dump(meta, f, indent=2)
    _meta_written_to = str(META_FILE)
except OSError as _write_err:
    if _write_err.errno == 107 or 'Transport endpoint' in str(_write_err):
        # Stale Drive mount. Fall back to /content/ (runtime only)
        _fallback = pathlib.Path('/content/_library_meta.json')
        try:
            with open(_fallback, 'w') as f:
                json.dump(meta, f, indent=2)
            _meta_written_to = str(_fallback)
            print('[scan] FALLBACK: _library_meta.json written to /content/ (runtime only).')
            print('[scan]   Your tags/favorites will NOT persist across Colab disconnects.')
            print('[scan]   To fix: re-run STEP 1 (force-remount Drive), then re-run STEP 2.')
            print('[scan]   You can then manually copy: cp /content/_library_meta.json <your_drive_path>/')
        except Exception as _fb_err:
            print(f'[scan] FALLBACK also failed: {_fb_err}')
            print('[scan]   No metadata was saved. Your tags will be lost on disconnect.')
    else:
        raise
if _meta_written_to:
    print(f'\n[scan] Metadata saved to {_meta_written_to}')
else:
    print('\n[scan] Metadata save FAILED. See warnings above.')
print(f'[scan] Thumbnail dir: {THUMB_DIR}')
print(f'[scan] Move on to STEP 3 (Gradio UI).')


In [ ]:
# @title STEP 3 — Gradio UI: Browse, Tag, Preview, Favorite
# @markdown The main browser. Gallery with thumbnails, filter sidebar,
# @markdown click-to-preview (3D mesh in `<model-viewer>`, 3DGS gets file info
# @markdown + 'convert to mesh' button), tag editor, favorite toggle. All state
# @markdown persists in the metadata sidecar JSON from STEP 2.
# @markdown
# @markdown **Tier filters** (populated automatically from `<slug>_meta.json`
# @markdown sidecars written by TripoSPlat STEP 8):
# @markdown - **Size class** filters by physical height: `small_prop` (<0.5m),
# @markdown   `prop` (0.5-2m, people/weapons), `tree_or_vehicle` (2-10m), `building` (>10m).
# @markdown - **Grounded only** shows assets that had the grounding transform
# @markdown   applied (sit on Y=0, centered in XZ, axis-aligned).

import os, sys, json, time, pathlib, shutil, traceback
import gradio as gr
from IPython.display import clear_output, display, HTML
import numpy as np
from PIL import Image
import trimesh
import open3d as o3d

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
THUMB_DIR = LIBRARY_DIR / '_thumbs'

if not META_FILE.exists():
    raise SystemExit(f'[ui] Run STEP 2 first ({META_FILE} not found).')
with open(META_FILE) as f:
    META = json.load(f)
print(f'[ui] Loaded metadata: {len(META["assets"])} assets')

# --- Helpers ----------------------------------------------------------
def _save_meta():
    META['updated'] = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()
    with open(META_FILE, 'w') as f:
        json.dump(META, f, indent=2)

def _all_tags():
    """Return the set of all tags used across all assets."""
    s = set()
    for m in META['assets'].values():
        for t in m.get('tags', []):
            s.add(t)
    return sorted(s)

def _all_formats():
    return sorted({m['format'] for m in META['assets'].values()})

def _all_size_classes():
    """Return the set of size_class values that exist (from meta.json sidecars)."""
    s = set()
    for m in META['assets'].values():
        sc = m.get('size_class')
        if sc:
            s.add(sc)
    return sorted(s)

def _list_assets(tag_filter=None, format_filter=None, favorite_only=False, search='',
                 size_class_filter=None, grounded_only=False):
    """Return list of (rel_path, thumbnail_or_None, format, size_mb) tuples
    matching the filters, sorted by path."""
    out = []
    for rel, m in META['assets'].items():
        if tag_filter and tag_filter not in m.get('tags', []):
            continue
        if format_filter and format_filter != m['format']:
            continue
        if favorite_only and not m.get('favorite', False):
            continue
        if size_class_filter and m.get('size_class') != size_class_filter:
            continue
        if grounded_only and not (m.get('grounding', {}) or {}).get('applied'):
            continue
        if search and search.lower() not in rel.lower():
            continue
        thumb = THUMB_DIR / (rel.replace('/', '__') + '.png')
        thumb_str = str(thumb) if thumb.exists() else None
        out.append((rel, thumb_str, m['format'], m['size_bytes'] / 1024 / 1024))
    out.sort(key=lambda x: x[0])

def _all_other_asset_choices(exclude_rel=""):
    """Return list of (filename, rel_path) tuples for all other assets.
    Used by the compare-B dropdown."""
    out = []
    for rel, m in META['assets'].items():
        if rel == exclude_rel:
            continue
        out.append((m.get('filename', rel), rel))
    out.sort()
    return out
    return out

def _preview_html(rel_path):
    """Build an HTML preview block. Mesh files: inline `<model-viewer>`.
    3DGS files: metadata card with note that 3DGS needs an external viewer.
    Images: inline `<img>`. Other: file info."""
    if rel_path is None:
        return ('<div style="font-family: sans-serif; max-width: 600px; padding: 40px; text-align: center; background: #f5f5f5; border-radius: 12px; color: #666;"><h3>No asset selected</h3><p>Click any thumbnail in the gallery above to preview it here.</p><p style="font-size: 0.9em;">Use the <b>Filter</b> sidebar on the left to narrow the gallery by tag, format, size class, etc.</p><p style="font-size: 0.85em; color: #999;">Keyboard: <kbd>←</kbd>/<kbd>→</kbd> to navigate, <kbd>?</kbd> for all shortcuts.</p></div>')
    m = META['assets'].get(rel_path, {})
    if not m:
        return f'<em>Unknown asset: {rel_path}</em>'
    full_path = LIBRARY_DIR / rel_path
    fmt = m.get('format', 'OTHER')
    size_mb = m.get('size_bytes', 0) / 1024 / 1024
    tags = ', '.join(m.get('tags', [])) or '(none)'
    favorite = 'YES' if m.get('favorite', False) else 'no'
    notes = m.get('notes', '') or '(no notes)'
    # Date formatting
    from datetime import datetime as _dt
    mtime_str = '(unknown)'
    if m.get('mtime'):
        try:
            mtime_str = _dt.fromtimestamp(m['mtime']).strftime('%Y-%m-%d %H:%M')
        except Exception:
            pass
    added_str = '(unknown)'
    if m.get('added'):
        try:
            added_str = m['added'][:10]
        except Exception:
            pass
    # Variant count badge
    n_var, _ = _variant_count(rel_path)
    variant_badge = ''
    if n_var > 1:
        view_count = m.get('view_count', 0)
        view_badge = ''
        if view_count:
            view_badge = f' | <b>Views:</b> {view_count}'
        variant_badge = f' | <b>Variants:</b> {n_var} (use Variant dropdown)' + view_badge
    # Tiering badge (from meta.json sidecar)
    sc = m.get('size_class')
    sc_badge = f' <span style="background: #4caf50; color: #fff; padding: 2px 8px; border-radius: 10px; font-size: 0.85em;">{sc}</span>' if sc else ''
    gr = m.get('grounding', {})
    gr_badge = ''
    if gr.get('applied'):
        ry = gr.get('rotation_y_degrees', 0)
        gr_badge = f' <span style="background: #1976d2; color: #fff; padding: 2px 8px; border-radius: 10px; font-size: 0.85em;">grounded {ry:+.0f}&deg;</span>'
    if fmt in ('MESH', 'COLLISION_MESH') and full_path.suffix.lower() in ('.glb', '.gltf', '.obj', '.fbx', '.ply', '.stl', '.3mf'):
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <model-viewer src="{full_path}" alt="{rel_path}" auto-rotate camera-controls style="width: 100%; height: 500px; background: #1a1a1a;"></model-viewer>
              <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js"></script>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}<br><b>Modified:</b> {mtime_str} | <b>Added to library:</b> {added_str}{variant_badge}</p>
            </div>
        '''
    elif fmt == 'IMAGE':
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <img src="{full_path}" style="max-width: 100%; max-height: 500px;">
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt in ('3DGS_RAW', '3DGS_PACKED', '3DGS_COMPRESSED'):
        # Per-format viewer links. SOG/GLB+KHR have direct PlayCanvas support;
        # others go to the community splat viewers.
        if fmt == '3DGS_COMPRESSED':
            ext = full_path.suffix.lower()
            if ext == '.sog':
                viewer_links = '<a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a> (native SOG support), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>'
                viewer_note = 'SOG is PlayCanvas-native (~10x compressed). Drop into [playcanvas.com/viewer](https://playcanvas.com/viewer/) for instant preview.'
            elif ext == '.spz':
                viewer_links = '<a href="https://scaniverse.com/" target="_blank">Niantic Scaniverse</a> (mobile), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a> (SPZ support), <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>'
                viewer_note = 'SPZ is the smallest cross-platform splat format (~5% of original PLY). Mobile-friendly via Scaniverse.'
            elif ext == '.ksplat':
                viewer_links = '<a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>'
                viewer_note = 'KSPLAT (mkkellogg format). Web-friendly, smaller than PLY.'
            elif ext == '.lcc':
                viewer_links = '<a href="https://lumalabs.ai/" target="_blank">LumaAI</a> (native LCC), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>'
                viewer_note = 'LCC (XGRIDS Luma format). LumaAI viewer has native support.'
            else:
                # GLB + KHR_gaussian_splatting
                viewer_links = '<a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a> (KHR_gaussian_splatting), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a> (Three.js plugin)'
                viewer_note = 'GLB with the new <code>KHR_gaussian_splatting</code> glTF extension. Future-proof standard — every engine will eventually support it.'
        else:
            # RAW PLY or PACKED SPLAT
            viewer_links = '<a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>, LumaAI'
            viewer_note = 'Raw 3DGS — 10x larger than compressed formats. Run <code>SplatTransform_Colab</code> to convert to SOG/SPZ/GLB for game use.'
        return f'''
            <div style="font-family: sans-serif; max-width: 800px; padding: 20px; background: #fff8e1; border-left: 4px solid #ff9800;">
              <h3>{rel_path}</h3>
              <p><b>3DGS Gaussian Splat ({fmt})</b> — cannot be displayed in <code>&lt;model-viewer&gt;</code>.
              Use a 3DGS viewer: {viewer_links}.</p>
              <p>{viewer_note}</p>
              <p><b>For textured visual mesh from this 3DGS:</b> use <a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb" target="_blank">SuGaR_Colab</a> (sharp, 2-3 hrs) or <a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb" target="_blank">GauStudio_Colab</a> (smooth, ~10 min).</p>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'LOD_META':
        try:
            content = full_path.read_text()
            pretty = content[:2000]  # truncate for display
        except Exception as e:
            return f'<em>Failed to read {rel_path}: {e}</em>'
        return f'''
            <div style="font-family: sans-serif; max-width: 800px; padding: 20px; background: #e3f2fd; border-left: 4px solid #1976d2;">
              <h3>{rel_path}</h3>
              <p><b>PlayCanvas LOD metadata</b> — sidecar for streamed SOG loading. Drop the parent folder
              (containing the .sog files + this <code>lod-meta.json</code>) into the
              <a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a> for progressive loading.</p>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
              <details><summary><b>Show lod-meta.json contents</b></summary>
              <pre style="background: #1a1a1a; color: #e0e0e0; padding: 12px; border-radius: 4px; max-height: 300px; overflow: auto;">{pretty}</pre>
              </details>
            </div>
        '''
    elif fmt == 'VIDEO':
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <video src="{full_path}" controls autoplay loop muted
                     style="max-width: 100%; max-height: 500px; background: #000;"></video>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'AUDIO':
        return f'''
            <div style="font-family: sans-serif; max-width: 800px; padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0;">
              <h3>{rel_path}</h3>
              <audio src="{full_path}" controls autoplay loop
                     style="width: 100%;"></audio>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'HDR':
        return f'''
            <div style="font-family: sans-serif; max-width: 800px; padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800;">
              <h3>{rel_path}</h3>
              <p><b>HDR environment map</b> — used as IBL (image-based lighting) source in PBR engines.
              Not directly viewable inline, but the <a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a>
              and <a href="https://threejs.org/editor/" target="_blank">Three.js editor</a> can preview it.</p>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'TEXT':
        try:
            content = full_path.read_text(errors='replace')
            pretty = content[:5000]
        except Exception as e:
            pretty = f'(failed to read: {e})'
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <details open><summary><b>Show contents (first 5KB)</b></summary>
              <pre style="background: #1a1a1a; color: #e0e0e0; padding: 12px; border-radius: 4px; max-height: 400px; overflow: auto;">{pretty}</pre>
              </details>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'META':
        try:
            content = full_path.read_text()
            pretty = content[:5000]
        except Exception as e:
            pretty = f'(failed to read: {e})'
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
              <details><summary><b>Show JSON / metadata</b></summary>
              <pre style="background: #1a1a1a; color: #e0e0e0; padding: 12px; border-radius: 4px; max-height: 400px; overflow: auto;">{pretty}</pre>
              </details>
            </div>
        '''
    else:
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <p>No inline preview for <code>{fmt}</code> ({full_path.suffix}).</p>
              <p><b>Size:</b> {size_mb:.2f} MB{sc_badge}{gr_badge}<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''

def _list_selected(checked_paths):
    """Return list of (rel, meta) for the asset paths the user has selected.
    Falls back to the currently-selected single asset if no boxes checked."""
    if not checked_paths:
        sel = gr_state.get('selected_path', '')
        if sel and sel in META['assets']:
            return [(sel, META['assets'][sel])]
        return []
    out = []
    for p in checked_paths:
        if p in META['assets']:
            out.append((p, META['assets'][p]))
    return out


def _variant_count(rel):
    """Count how many related files (same slug, different extensions) exist
    in the library, including the current asset."""
    if not rel:
        return 0, []
    p = LIBRARY_DIR / rel
    slug = p.stem
    parent = p.parent
    if not parent.exists():
        return 1, [p.suffix]
    variants = sorted([f.suffix for f in parent.iterdir() if f.is_file() and f.stem == slug])
    return len(variants), variants


def _build_gallery(sort_by='name', sort_asc=True, group_by='none'):
    """Build gallery with optional sort + group. Sort keys: name, size, size_class,
    grounding, format. Group keys: none, format, size_class, grounding, folder."""
    assets = _list_assets(
        tag_filter=gr_state.get('tag'),
        format_filter=gr_state.get('format'),
        favorite_only=gr_state.get('fav'),
        size_class_filter=gr_state.get('size_class'),
        grounded_only=gr_state.get('grounded'),
        search=gr_state.get('search', ''),
        untagged_only=gr_state.get('untagged', False),
        has_colliders_only=gr_state.get('has_colliders', False),
        recent_days=gr_state.get('recent_days', 0),
    )
    # Sort
    sort_keys = {
        'name': lambda r: r[0].lower(),
        'size': lambda r: -r[1]['size_bytes'] if sort_asc else r[1]['size_bytes'],
        'size_class': lambda r: (r[1].get('size_class') or 'zzz', r[0].lower()),
        'grounding': lambda r: ((r[1].get('grounding') or {}).get('applied', False), r[0].lower()),
        'format': lambda r: (r[1].get('format', 'OTHER'), r[0].lower()),
        'mtime': lambda r: -r[1].get('mtime', 0) if sort_asc else r[1].get('mtime', 0),
    }
    if sort_by in sort_keys:
        try:
            assets = sorted(assets, key=sort_keys[sort_by], reverse=not sort_asc and sort_by != 'name')
        except Exception:
            pass
    if group_by == 'none':
        return assets
    # Group: prepend a header row for each group
    from collections import defaultdict
    groups = defaultdict(list)
    for a in assets:
        rel, m, _fmt, _sz = a
        if group_by == 'format':
            key = m.get('format', 'OTHER')
        elif group_by == 'size_class':
            key = m.get('size_class') or '(no size_class)'
        elif group_by == 'grounding':
            key = 'grounded' if (m.get('grounding') or {}).get('applied') else 'not grounded'
        elif group_by == 'folder':
            key = str(pathlib.Path(rel).parent)
        else:
            key = 'all'
        groups[key].append(a)
    out = []
    for key in sorted(groups.keys()):
        out.append((None, f'__GROUP__:{key} ({len(groups[key])})', None, 0))
        out.extend(groups[key])
    return out


def _batch_apply_tag(checked_paths, tag_text):
    """Apply a tag to all selected assets."""
    selected = _list_selected(checked_paths)
    tag = tag_text.strip()
    if not tag or not selected:
        return gr.Info('Pick assets and enter a tag first.'), [], ''
    count = 0
    for _rel, m in selected:
        tags = m.get('tags', [])
        if tag not in tags:
            tags.append(tag)
            count += 1
    _save_meta()
    return gr.Info(f'Added tag "{tag}" to {count} asset(s).'), [], tag


def _batch_set_tier(checked_paths, tier):
    """Set the size_class (acts as our tier label) on all selected assets."""
    selected = _list_selected(checked_paths)
    if not selected or tier not in ('small_prop', 'prop', 'tree_or_vehicle', 'building'):
        return gr.Info('Pick assets and choose a tier first.'), []
    for _rel, m in selected:
        m['size_class'] = tier
    _save_meta()
    return gr.Info(f'Set tier to "{tier}" on {len(selected)} asset(s).'), []


def _batch_delete_from_meta(checked_paths):
    """Remove selected assets from the library meta (does NOT delete files)."""
    selected = _list_selected(checked_paths)
    if not selected:
        return gr.Info('Pick assets first.'), []
    for rel, _m in selected:
        META['assets'].pop(rel, None)
    _save_meta()
    return gr.Info(f'Removed {len(selected)} asset(s) from library meta. Files untouched.'), []

def _save_preset(name, filters):
    """Save current filter state as a named preset."""
    if not name or not name.strip():
        return gr.Warning('Preset name required.')
    META.setdefault('presets', {})[name.strip()] = dict(filters)
    _save_meta()
    return gr.Info(f'Saved preset: {name}')

def _load_preset(name):
    """Return the filter dict for a saved preset, or None."""
    if not name or name not in META.get('presets', {}):
        return None
    return META['presets'][name]

def _delete_preset(name):
    """Remove a saved preset."""
    if name in META.get('presets', {}):
        del META['presets'][name]
        _save_meta()
        return gr.Info(f'Deleted preset: {name}')

def _compare_stats_html(a_rel, b_rel):
    """Build a side-by-side comparison stats table for two assets."""
    a = META['assets'].get(a_rel, {})
    b = META['assets'].get(b_rel, {})
    if not a or not b:
        return '<em>Both assets must be selected.</em>'
    from datetime import datetime as _dt
    def fmt_dt(ts):
        if not ts:
            return '(unknown)'
        try:
            return _dt.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M')
        except Exception:
            return '(unknown)'
    def fmt_tags(m):
        tags = m.get('tags', [])
        return ', '.join(tags) if tags else '(none)'
    def fmt_size(b):
        return f"{b/1024/1024:.2f} MB" if b else '0'
    a_gr = (a.get('grounding') or {}).get('applied', False)
    b_gr = (b.get('grounding') or {}).get('applied', False)
    rows = [
        ('Path', a_rel, b_rel),
        ('Format', a.get('format', '?'), b.get('format', '?')),
        ('Size', fmt_size(a.get('size_bytes', 0)), fmt_size(b.get('size_bytes', 0))),
        ('Size class', a.get('size_class') or '(none)', b.get('size_class') or '(none)'),
        ('Grounded', 'YES' if a_gr else 'no', 'YES' if b_gr else 'no'),
        ('Favorite', 'YES' if a.get('favorite') else 'no', 'YES' if b.get('favorite') else 'no'),
        ('Views', a.get('view_count', 0), b.get('view_count', 0)),
        ('Tags', fmt_tags(a), fmt_tags(b)),
        ('Modified', fmt_dt(a.get('mtime')), fmt_dt(b.get('mtime'))),
    ]
    html = "<div style=\"font-family: sans-serif; max-width: 900px;\"><table style=\"border-collapse: collapse; width: 100%;\">"
    html += '<tr style="background: #bbdefb;"><th style="text-align: left; padding: 8px;">Field</th><th style="text-align: left; padding: 8px;">A: ' + a_rel + '</th><th style="text-align: left; padding: 8px;">B: ' + b_rel + '</th></tr>'
    for k, av, bv in rows:
        # Highlight diffs
        diff = av != bv
        bg = '#fff3e0' if diff else 'transparent'
        html += f'<tr style="background: {bg};"><td style="padding: 6px; border-bottom: 1px solid #eee;"><b>{k}</b></td><td style="padding: 6px; border-bottom: 1px solid #eee;">{av}</td><td style="padding: 6px; border-bottom: 1px solid #eee;">{bv}</td></tr>'
    html += '</table></div>'
    return html

def _zip_export_assets(checked_paths, include_sidecars=True, include_manifest=True, output_dir='/content/'):
    """Pack selected assets into a single ZIP file.

    Args:
      checked_paths: list of rel paths from batch_assets_cg CheckboxGroup.
                        Falls back to the current gallery filter if empty.
      include_sidecars: include <slug>_meta.json sidecars for splat assets.
      include_manifest: include a <timestamp>_manifest.csv with per-asset metadata.
      output_dir: where to write the ZIP file. Default /content/ (Colab working dir).

    Returns: (zip_path, status_html, gr.File update)
    """
    import zipfile, csv, time
    from datetime import datetime as _dt
    if not checked_paths:
        # Fall back to current filter
        checked_paths = [a[0] for a in _list_assets(
            tag_filter=gr_state.get('tag'),
            format_filter=gr_state.get('format'),
            favorite_only=gr_state.get('fav'),
            size_class_filter=gr_state.get('size_class'),
            grounded_only=gr_state.get('grounded'),
            search=gr_state.get('search', ''),
            untagged_only=gr_state.get('untagged', False),
            has_colliders_only=gr_state.get('has_colliders', False),
            recent_days=gr_state.get('recent_days', 0))
            if a[0] is not None]
    if not checked_paths:
        return None, '<p style="color: #c62828;">No assets to export.</p>', gr.update(visible=False)
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    zip_name = f'aei_library_export_{ts}.zip'
    zip_path = pathlib.Path(output_dir) / zip_name
    n_files = 0
    n_sidecars = 0
    n_skipped = 0
    n_bytes = 0
    manifest_rows = []
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for rel in checked_paths:
            if rel not in META['assets']:
                n_skipped += 1
                continue
            p = LIBRARY_DIR / rel
            if not p.exists():
                n_skipped += 1
                continue
            # Add the main file
            try:
                zf.write(p, arcname=rel)
                n_files += 1
                n_bytes += p.stat().st_size
            except Exception as e:
                print(f'[zip] WARN: {rel}: {e}')
                n_skipped += 1
                continue
            # Add sidecar meta.json if requested and present
            if include_sidecars:
                sidecar = p.parent / f'{p.stem}_meta.json'
                if sidecar.exists():
                    try:
                        zf.write(sidecar, arcname=str(sidecar.relative_to(LIBRARY_DIR)))
                        n_sidecars += 1
                    except Exception:
                        pass
            # Add to manifest
            m = META['assets'][rel]
            manifest_rows.append({
                'path': rel,
                'filename': p.name,
                'format': m.get('format', '?'),
                'size_bytes': p.stat().st_size,
                'size_class': m.get('size_class', ''),
                'grounded': (m.get('grounding') or {}).get('applied', False),
                'favorite': m.get('favorite', False),
                'view_count': m.get('view_count', 0),
                'tags': ','.join(m.get('tags', [])),
                'notes': m.get('notes', '').replace('\n', ' ').replace('\r', '')[:200],
                'mtime': _dt.fromtimestamp(m['mtime']).isoformat() if m.get('mtime') else '',
            })
        # Add manifest as the last file
        if include_manifest and manifest_rows:
            import io
            buf = io.StringIO()
            w = csv.DictWriter(buf, fieldnames=list(manifest_rows[0].keys()))
            w.writeheader()
            for row in manifest_rows:
                w.writerow(row)
            manifest_text = buf.getvalue()
            zf.writestr(f'aei_manifest_{ts}.csv', manifest_text)
    size_mb = zip_path.stat().st_size / 1024 / 1024
    status_html = (
        f'<div style="background: #e8f5e9; padding: 12px; border-radius: 8px;">'
        f'<b>Exported {n_files} assets</b> ({size_mb:.2f} MB ZIP, {n_bytes/1024/1024:.1f} MB uncompressed)<br>'
        f'Sidecars: {n_sidecars} | Manifest: {("yes" if include_manifest else "no")} | Skipped: {n_skipped}<br>'
        f'<small>Path: <code>{zip_path}</code></small>'
        f'</div>'
    )
    return str(zip_path), status_html, gr.update(value=str(zip_path), visible=True)

def _batch_rename(checked_paths, mode, text):
    """Rename selected assets in place. Modes: prepend, append, replace.
    For safety, this renames the FILES on disk + updates META[rel].
    Returns count renamed."""
    selected = _list_selected(checked_paths)
    if not selected or not text:
        return 0, gr.Info('Pick assets and enter text.')
    n = 0
    for rel, m in selected:
        p = LIBRARY_DIR / rel
        if not p.exists():
            continue
        stem, ext = p.stem, p.suffix
        if mode == 'prepend':
            new_stem = text + stem
        elif mode == 'append':
            new_stem = stem + text
        elif mode == 'replace':
            # Replace any occurrence of the text in the stem (literal)
            # Format: "old_text|new_text"
            if '|' in text:
                old_t, new_t = text.split('|', 1)
                new_stem = stem.replace(old_t, new_t)
            else:
                return 0, gr.Info('Replace mode: use old|new format.')
        else:
            return 0, gr.Warning(f'Unknown rename mode: {mode}')
        new_name = new_stem + ext
        new_path = p.parent / new_name
        if new_path.exists():
            continue
        try:
            p.rename(new_path)
            # Update META
            new_rel = str(new_path.relative_to(LIBRARY_DIR))
            m['path'] = new_rel
            m['filename'] = new_name
            META['assets'][new_rel] = m
            META['assets'].pop(rel, None)
            n += 1
        except Exception as e:
            print(f'[rename] WARN: {rel}: {e}')
    _save_meta()
    return n, gr.Info(f'Renamed {n} asset(s). Re-run STEP 2 to refresh scan.')
    return gr.update()

def _list_presets():
    """Return list of saved preset names."""
    return sorted(META.get('presets', {}).keys())


# JavaScript for keyboard shortcuts (injected via gr.HTML)
KEYBOARD_JS = """
<script>
(function() {
  function findBtn(testid) {
    return document.querySelector(`[data-testid="${testid}"]`);
  }
  document.addEventListener('keydown', function(e) {
    // Skip if typing in an input
    const tag = e.target.tagName;
    if (tag === 'INPUT' || tag === 'TEXTAREA' || e.target.isContentEditable) return;
    if (e.key === 'ArrowLeft')  { const b = findBtn('prev-btn');  if (b) b.click(); }
    if (e.key === 'ArrowRight') { const b = findBtn('next-btn'); if (b) b.click(); }
    if (e.key === ' ')          { const b = findBtn('fav-toggle'); if (b) { b.click(); e.preventDefault(); } }
    if (e.key === '?')          { const b = findBtn('help-btn'); if (b) b.click(); }
    if (e.key.toLowerCase() === 'f') { const b = findBtn('search-box'); if (b) b.focus(); }
    if (e.key.toLowerCase() === 'n') { const b = findBtn('next-btn'); if (b) b.click(); }
    if (e.key.toLowerCase() === 'p') { const b = findBtn('prev-btn'); if (b) b.click(); }
    if (e.key.toLowerCase() === 't') { const b = findBtn('cycle-tier-btn'); if (b) b.click(); }
    if (e.key.toLowerCase() === 'd') { const b = findBtn('compare-btn'); if (b) b.click(); }
  });
})();
</script>
"""

gr_state = {'tag': None, 'format': None, 'fav': False, 'search': '',
            'size_class': None, 'grounded': False,
            'untagged': False, 'has_colliders': False, 'recent_days': 0,
            'selected_path': '', 'preview_mode': 'split',
            'compare_b': ''}

# --- Gradio UI ----------------------------------------------------------
with gr.Blocks(title='AEI Asset Library Browser', theme=gr.themes.Soft(),
               css='footer {visibility: hidden}') as demo:
    gr.Markdown('# AEI Asset Library Browser\n'
                f'Library: `{LIBRARY_DIR}` — {len(META["assets"])} assets')
    gr.HTML(KEYBOARD_JS)

    # Top toolbar (count + sort + group + fullscreen toggle + help)
    with gr.Row():
        count_md = gr.Markdown(value=f'**{len(META["assets"])} assets loaded.** Use the filter sidebar on the left.')
        sort_dd = gr.Dropdown(choices=['name', 'size', 'size_class', 'grounding', 'format', 'mtime'],
                            value='name', label='Sort by', scale=0, min_width=100,
                            info='Sort the gallery by this field.')
        sort_asc_cb = gr.Checkbox(value=True, label='Asc', scale=0,
                                  info='Ascending order. Uncheck for descending.')
        group_dd = gr.Dropdown(choices=['none', 'format', 'size_class', 'grounding', 'folder'],
                             value='none', label='Group by', scale=0, min_width=120,
                             info='Insert header rows in the gallery to group assets by this field.')
        fullscreen_btn = gr.Button('⛶ Full preview', scale=0, elem_id='fullscreen-btn')
        help_btn = gr.Button('? Shortcuts', scale=0, elem_id='help-btn')
    # Hidden help state
    show_help = gr.State(value=False)

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown('### Filter')
            search_box = gr.Textbox(value='', label='Search filename', elem_id='search-box',
                                     placeholder='e.g. hero, glb, knight...', info='Filter by filename substring. Press F to focus.')
            tag_dd = gr.Dropdown(choices=[None] + _all_tags(), value=None,
                                  label='Filter by tag', info='Show only assets with this tag.')
            fmt_dd = gr.Dropdown(choices=[None] + _all_formats(), value=None,
                                  label='Filter by format', info='MESH, 3DGS_RAW, IMAGE, etc.')
            size_dd = gr.Dropdown(choices=[None] + _all_size_classes(), value=None,
                                   label='Filter by size class',
                                   info='small_prop / prop / tree_or_vehicle / building. Populated from <slug>_meta.json sidecars.')
            grounded_cb = gr.Checkbox(value=False, label='Grounded only',
                                       info='Show only assets that had the grounding transform applied.')
            fav_cb = gr.Checkbox(value=False, label='Favorites only', info='Show only assets you starred.')
            # New smart filters
            untagged_cb = gr.Checkbox(value=False, label='Untagged only',
                                      info='Show only assets that have no tags yet. Useful for prioritizing tagging work.')
            has_colliders_cb = gr.Checkbox(value=False, label='Has colliders',
                                            info='Show only assets that have <slug>_hull.glb or <slug>_collision.glb.')
            recent_sl = gr.Slider(0, 365, value=0, step=1, label='Modified in last N days',
                                    info='0 = all. Filter to assets modified in the last N days.')
            gr.Markdown('### Batch actions (pick assets below)')
            batch_assets_cg = gr.CheckboxGroup(choices=[], value=[], label='Selected for batch',
                                                info='Check assets to operate on multiple at once.')
            batch_tag_tb = gr.Textbox(value='', label='Tag to add',
                                        placeholder='e.g. hero, vehicle',
                                        info='Add this tag to all selected assets.')
            batch_tag_btn = gr.Button('+ Add tag to selected', variant='primary')
            batch_tier_dd = gr.Dropdown(choices=['small_prop', 'prop', 'tree_or_vehicle', 'building'],
                                         label='Set tier (size_class)',
                                         info='Tag the selected assets with a size_class. Used for procedural placement.')
            batch_tier_btn = gr.Button('Set tier on selected')
            batch_delete_btn = gr.Button('Remove from library meta', variant='stop')
            gr.Markdown('### Rename selected')
            batch_rename_mode = gr.Radio(choices=['prepend', 'append', 'replace'], value='prepend', label='Mode',
                                         info='prepend: add text before stem. append: add text after. replace: use old|new format.')
            batch_rename_tb = gr.Textbox(value='', label='Text (or old|new for replace)',
                                          placeholder='e.g. hero_ (prepend) or old|new (replace)',
                                          info='Mode determines how the text is used.')
            batch_rename_btn = gr.Button('Rename files (moves on disk!)', variant='stop')
            gr.Markdown('### Export selected as ZIP')
            export_status_html = gr.HTML(value='<em>No export yet. Pick assets below and click Export.</em>')
            export_file = gr.File(value=None, label='Download ZIP', visible=False,
                                   info='When an export completes, the ZIP appears here. Click to download.')
            export_selected_btn = gr.Button('Export CHECKED to ZIP', variant='primary')
            export_filtered_btn = gr.Button('Export all currently-filtered to ZIP')
            gr.Markdown('### Saved filter presets')
            preset_dd = gr.Dropdown(choices=_list_presets(), value=None, label='Load preset',
                                       info='Pick a saved filter preset to restore.')
            preset_load_btn = gr.Button('Load preset')
            preset_name_tb = gr.Textbox(value='', label='Save current as...',
                                          placeholder='e.g. heroes-only',
                                          info='Name for the current filter state. Saved to _library_meta.json["presets"].')
            preset_save_btn = gr.Button('Save current filter as preset')
            preset_delete_btn = gr.Button('Delete selected preset', variant='stop')
            gr.Markdown('### Edit selected asset')
            sel_path = gr.Textbox(value='', label='Selected path',
                                   interactive=False, info='Auto-populated on gallery click.')
            tags_box = gr.Textbox(value='', label='Tags (comma-separated)',
                                     placeholder='hero, vehicle, low-poly',
                                     info='Comma-separated. Used by Filter + exports.')
            notes_box = gr.Textbox(value='', label='Notes',
                                      placeholder='description, source, etc.',
                                      lines=3, info='Shown in preview + HTML portfolio.')
            siblings_dd = gr.Dropdown(choices=[], value=None, label='Variant (same slug, other format)',
                                       info='Sibling files: same name, different extension. Click to switch.')
            prev_btn = gr.Button('← Previous (\u2190)', elem_id='prev-btn')
            next_btn = gr.Button('Next (\u2192) >>', elem_id='next-btn', variant='primary')
            fav_btn = gr.Button('Toggle favorite (Space)', elem_id='fav-toggle')
            cycle_tier_btn = gr.Button('Cycle tier (T)', elem_id='cycle-tier-btn')
            save_btn = gr.Button('Save changes', variant='primary')

        with gr.Column(scale=3):
            gallery = gr.Gallery(value=_build_gallery(), columns=4, height=600,
                                 label='Assets (click to preview)',
                                 allow_preview=False)
            preview = gr.HTML(value=_preview_html(None), label='Preview')

    # Fullscreen preview (a separate HTML shown in a Column that's normally hidden)
    with gr.Column(visible=False, elem_id='fullscreen-col') as fullscreen_col:
        fullscreen_html = gr.HTML(value='<em>Select an asset and click ⛶ Full preview to expand.</em>')
        fullscreen_close_btn = gr.Button('Close fullscreen (\u26F6)')

    with gr.Column(visible=False, elem_id='help-modal') as help_modal:
        gr.HTML(value="""<div style="font-family: sans-serif; padding: 24px; background: linear-gradient(135deg, #e3f2fd 0%, #f3e5f5 100%); border-radius: 12px;"><h2>AEI Asset Library Browser - Help</h2><h3>Keyboard Shortcuts</h3><ul><li><kbd>←</kbd> / <kbd>→</kbd> - Previous / next asset in current filter</li><li><kbd>Space</kbd> - Toggle favorite on selected asset</li><li><kbd>P</kbd> / <kbd>N</kbd> - Previous / next (alias for arrows)</li><li><kbd>T</kbd> - Cycle size_class tier on selected</li><li><kbd>D</kbd> - Open compare modal (current asset vs picked one)</li><li><kbd>F</kbd> - Focus the search box</li><li><kbd>?</kbd> - Show this help</li></ul><h3>Filters</h3><ul><li><b>Search</b> - filename substring (case-insensitive)</li><li><b>Tag</b> - user-applied tags (comma-separated)</li><li><b>Format</b> - MESH / 3DGS_RAW / 3DGS_PACKED / 3DGS_COMPRESSED / IMAGE / VIDEO / AUDIO / HDR / TEXT / META / ARCHIVE</li><li><b>Size class</b> - small_prop / prop / tree_or_vehicle / building (from <code>&lt;slug&gt;_meta.json</code> sidecars)</li><li><b>Grounded only</b> - had grounding transform applied</li><li><b>Favorites only</b> - starred assets</li><li><b>Untagged only</b> - prioritize tagging work</li><li><b>Has colliders</b> - filter to <code>&lt;slug&gt;_hull.glb</code> or <code>&lt;slug&gt;_collision.glb</code></li><li><b>Modified in last N days</b> - 0 = all, 7 = week, 30 = month</li></ul><h3>Sort + Group</h3><ul><li><b>Sort by</b> - name / size / size_class / grounding / format / mtime</li><li><b>Group by</b> - insert header rows in gallery: format / size_class / grounding / folder</li></ul><h3>Batch Actions</h3><p>Use the <b>Selected for batch</b> checkbox group to pick multiple assets, then:</p><ul><li><b>Add tag to selected</b></li><li><b>Set tier (size_class)</b> - small_prop / prop / tree_or_vehicle / building</li><li><b>Remove from library meta</b> - files untouched</li></ul><h3>Variant Switching</h3><p>When an asset has siblings (same slug, different extension), the <b>Variant</b> dropdown appears. Click to switch preview.</p><h3>Workflow</h3><p>1. STEP 2 scans library and builds <code>_library_meta.json</code>.<br>2. STEP 3 to browse, tag, favorite.<br>3. <code>&lt;slug&gt;_meta.json</code> sidecars from TripoSPlat STEP 8 and SplatTransform STEP 6 auto-populate <code>size_class</code> + <code>grounding</code>.<br>4. Tags + favorites + tiers persist in <code>_library_meta.json</code>.</p></div>""")

    # Compare modal: side-by-side preview of two assets
    with gr.Column(visible=False, elem_id='compare-modal') as compare_modal:
        gr.Markdown('## Compare two assets')
        compare_b_dd = gr.Dropdown(choices=[], value=None, label='Compare with...',
                                    info='Pick another asset to compare side-by-side with the current selection.')
        with gr.Row():
            compare_a_html = gr.HTML(value='<em>Select asset A first.</em>')
            compare_b_html = gr.HTML(value='<em>Pick asset B from the dropdown above.</em>')
        compare_stats_html = gr.HTML(value='<em>Comparison stats will appear here.</em>')
        compare_close_btn = gr.Button('Close compare (Esc)')
        help_modal_close_btn = gr.Button('Close help (Esc)')

    def on_select(evt: gr.SelectData):
        chosen = evt.value
        if chosen is None:
            gr_state['selected_path'] = ''
        # Increment view count
        if rel and rel in META['assets']:
            META['assets'][rel]['view_count'] = META['assets'][rel].get('view_count', 0) + 1
            _save_meta()
            return '', '', '', _preview_html(None), gr.update(choices=[])
        thumb_path = pathlib.Path(chosen)
        rel = thumb_path.stem.replace('__', '/')
        gr_state['selected_path'] = rel
        m = META['assets'].get(rel, {})
        if not m:
            return rel, '', '', _preview_html(rel), gr.update(choices=[])
        # Build sibling list (same slug, all extensions) for fast variant switching
        n_variants, variants = _variant_count(rel)
        full_path = LIBRARY_DIR / rel
        parent = full_path.parent
        if parent.exists() and n_variants > 1:
            siblings = []
            for f in sorted(parent.iterdir()):
                if f.is_file() and f.stem == full_path.stem and f.suffix:
                    siblings.append((f.name, str(f.relative_to(LIBRARY_DIR))))
            sibling_choices = [(name, rel_path) for name, rel_path in siblings]
        else:
            sibling_choices = []
        return (rel, ', '.join(m.get('tags', [])), m.get('notes', ''),
                _preview_html(rel), gr.update(choices=sibling_choices, value=[]))

    def on_search(s, t, f, sc, grd, fav, untagged, has_coll, recent):
        gr_state['search'] = s
        gr_state['tag'] = t
        gr_state['format'] = f
        gr_state['size_class'] = sc
        gr_state['grounded'] = grd
        gr_state['fav'] = fav
        gr_state['untagged'] = untagged
        gr_state['has_colliders'] = has_coll
        gr_state['recent_days'] = int(recent)
        assets = _list_assets(tag_filter=t, format_filter=f, favorite_only=fav, size_class_filter=sc, grounded_only=grd, search=s, untagged_only=untagged, has_colliders_only=has_coll, recent_days=int(recent))
        total = len(META['assets'])
        visible = sum(1 for a in assets if a[0] is not None)
        count_text = f'Showing **{visible}** of **{total}** assets'
        any_filter = any([s, t, f, sc, grd, fav, untagged, has_coll, recent])
        if any_filter and visible == 0 and total > 0:
            return assets, count_text, gr.Warning('No assets match the current filter. Try clearing it.')
        elif any_filter and visible < total:
            return assets, count_text, gr.Info(f'Filter: {visible} of {total} assets shown')
        return assets, count_text, gr.update()
        gr_state['search'] = s
        gr_state['tag'] = t
        gr_state['format'] = f
        gr_state['size_class'] = sc
        gr_state['grounded'] = grd
        gr_state['fav'] = fav
        gr_state['untagged'] = untagged
        gr_state['has_colliders'] = has_coll
        gr_state['recent_days'] = int(recent)
        return _list_assets(tag_filter=t, format_filter=f,
                            favorite_only=fav, size_class_filter=sc,
                            grounded_only=grd, search=s,
                            untagged_only=untagged,
                            has_colliders_only=has_coll,
                            recent_days=int(recent))

    def on_sort_change(sort_by, sort_asc, group_by):
        gr_state['sort_by'] = sort_by
        gr_state['sort_asc'] = sort_asc
        gr_state['group_by'] = group_by
        assets = _build_gallery(sort_by=sort_by, sort_asc=sort_asc, group_by=group_by)
        visible = sum(1 for a in assets if a[0] is not None)
        total = len(META['assets'])
        return assets, f'Showing **{visible}** of **{total}** assets'

    def on_save(rel, tags_str, notes):
        if not rel or rel not in META['assets']:
            return gr.update(), gr.update(value='(nothing to save)')
        m = META['assets'][rel]
        m['tags'] = [t.strip() for t in tags_str.split(',') if t.strip()]
        m['notes'] = notes
        _save_meta()
        return _list_assets(tag_filter=gr_state.get('tag'),
                            format_filter=gr_state.get('format'),
                            favorite_only=gr_state.get('fav'),
                            size_class_filter=gr_state.get('size_class'),
                            grounded_only=gr_state.get('grounded'),
                            search=gr_state.get('search', '')), \
               gr.update(value=f'Saved {rel} (tags={m["tags"]})')

    def on_fav(rel):
        if not rel or rel not in META['assets']:
            return gr.update(), gr.update(value='(select an asset first)')
        m = META['assets'][rel]
        m['favorite'] = not m.get('favorite', False)
        _save_meta()
        return _list_assets(tag_filter=gr_state.get('tag'),
                            format_filter=gr_state.get('format'),
                            favorite_only=gr_state.get('fav'),
                            size_class_filter=gr_state.get('size_class'),
                            grounded_only=gr_state.get('grounded'),
                            search=gr_state.get('search', '')), \
               gr.update(value=f'{"FAVORITED" if m["favorite"] else "unfavorited"} {rel}')

    def on_prev_next(direction):
        """Navigate to previous (-1) or next (+1) asset in the current gallery."""
        current_path = gr_state.get('selected_path', '')
        assets = _list_assets(tag_filter=gr_state.get('tag'),
                            format_filter=gr_state.get('format'),
                            favorite_only=gr_state.get('fav'),
                            size_class_filter=gr_state.get('size_class'),
                            grounded_only=gr_state.get('grounded'),
                            search=gr_state.get('search', ''),
                            untagged_only=gr_state.get('untagged', False),
                            has_colliders_only=gr_state.get('has_colliders', False),
                            recent_days=gr_state.get('recent_days', 0))
        if not assets:
            return '', '', '', _preview_html(None), gr.update(choices=[])
        paths = [a[0] for a in assets]
        if current_path in paths:
            idx = paths.index(current_path)
        else:
            idx = 0
        new_idx = (idx + direction) % len(paths)
        new_path = paths[new_idx]
        gr_state['selected_path'] = new_path
        m = META['assets'].get(new_path, {})
        # Build sibling list
        full_path = LIBRARY_DIR / new_path
        parent = full_path.parent
        if parent.exists():
            siblings = []
            for f in sorted(parent.iterdir()):
                if f.is_file() and f.stem == full_path.stem and f.suffix:
                    siblings.append((f.name, f.name))
            sibling_choices = [(name, name) for name, _ in siblings]
        else:
            sibling_choices = []
        return (new_path, ', '.join(m.get('tags', [])),
                m.get('notes', ''), _preview_html(new_path),
                gr.update(choices=sibling_choices, value=[]))

    def on_fullscreen():
        """Show the fullscreen preview column with the current asset."""
        rel = gr_state.get('selected_path', '')
        if not rel:
            return gr.update(visible=False), gr.update(value='<em>Select an asset first.</em>')
        return gr.update(visible=True), gr.update(value=_preview_html(rel))

    def on_fullscreen_close():
        return gr.update(visible=False), gr.update()

    def on_save_preset(name, s, t, f, sc, grd, fav, untagged, has_coll, recent):
        """Save current filter state as named preset."""
        filters = {
            'search': s, 'tag': t, 'format': f, 'size_class': sc,
            'grounded': grd, 'fav': fav, 'untagged': untagged,
            'has_colliders': has_coll, 'recent_days': int(recent),
        }
        return _save_preset(name, filters)

    def on_load_preset(name):
        """Load a saved preset into gr_state and refresh gallery + count."""
        preset = _load_preset(name)
        if preset is None:
            return ([], '', None, None, False, False, False, False, 0, [],[], gr.update(), gr.update())
        # Apply preset to gr_state and re-list
        for k, v in preset.items():
            gr_state[k] = v
        assets = _list_assets(tag_filter=preset.get('tag'),
                               format_filter=preset.get('format'),
                               favorite_only=preset.get('fav', False),
                               size_class_filter=preset.get('size_class'),
                               grounded_only=preset.get('grounded', False),
                               search=preset.get('search', ''),
                               untagged_only=preset.get('untagged', False),
                               has_colliders_only=preset.get('has_colliders', False),
                               recent_days=preset.get('recent_days', 0))
        visible = sum(1 for a in assets if a[0] is not None)
        total = len(META['assets'])
        count_text = f'Showing **{visible}** of **{total}** assets (preset: {name})'
        return (preset.get('search', ''), preset.get('tag'),
                preset.get('format'), preset.get('size_class'),
                preset.get('grounded', False), preset.get('fav', False),
                preset.get('untagged', False), preset.get('has_colliders', False),
                preset.get('recent_days', 0), assets, count_text,
                gr.update(choices=_list_presets(), value=name), gr.Info(f'Loaded preset: {name}'))

    def on_delete_preset(name):
        """Delete the selected preset and refresh the dropdown."""
        result = _delete_preset(name)
        return gr.update(choices=_list_presets(), value=None), result

    def on_cycle_tier(rel):
        """Cycle through small_prop → prop → tree_or_vehicle → building → (none) → small_prop."""
        if not rel or rel not in META['assets']:
            return gr.update(), gr.update(value='(select an asset first)')
        m = META['assets'][rel]
        order = [None, 'small_prop', 'prop', 'tree_or_vehicle', 'building']
        cur = m.get('size_class')
        try:
            idx = order.index(cur) if cur in order else -1
        except ValueError:
            idx = -1
        nxt = order[(idx + 1) % len(order)]
        m['size_class'] = nxt
        _save_meta()
        new_label = nxt or '(no tier)'
        return _list_assets(tag_filter=gr_state.get('tag'),
                            format_filter=gr_state.get('format'),
                            favorite_only=gr_state.get('fav'),
                            size_class_filter=gr_state.get('size_class'),
                            grounded_only=gr_state.get('grounded'),
                            search=gr_state.get('search', '')), \
               gr.update(value=f'Tier of {rel}: {new_label}')

    def on_help():
        """Show keyboard shortcuts help."""
        help_html = """
        <div style="font-family: sans-serif; padding: 16px; background: #e3f2fd; border-radius: 8px;">
        <h3>Keyboard Shortcuts</h3>
        <table style="border-collapse: collapse;">
        <tr><td><kbd>←</kbd> / <kbd>→</kbd></td><td>Previous / next asset in current filter</td></tr>
        <tr><td><kbd>Space</kbd></td><td>Toggle favorite on selected asset</td></tr>
        <tr><td><kbd>F</kbd></td><td>Focus the search box</td></tr>
        <tr><td><kbd>P</kbd> / <kbd>N</kbd></td><td>Previous / next (alias for arrows)</td></tr>
        <tr><td><kbd>T</kbd></td><td>Cycle size_class tier on selected</td></tr>
        <tr><td><kbd>?</kbd></td><td>Show this help</td></tr>
        </table>
        <p style="margin-top: 8px;">Shortcuts are inactive when typing in a text field.</p>
        </div>
        """
        return gr.update(value=help_html)

    def on_help_modal():
        """Show the full help modal."""
        return gr.update(visible=True)

    def on_help_modal_close():
        """Close the help modal."""
        return gr.update(visible=False)

    def on_compare():
        """Open the compare modal with the current asset as A."""
        rel = gr_state.get('selected_path', '')
        if not rel or rel not in META['assets']:
            return (gr.update(visible=False),
                    gr.update(value='<em>Select an asset first.</em>'),
                    gr.update(value='<em>Pick asset B from the dropdown above.</em>'),
                    gr.update(value='<em>No asset A selected.</em>'),
                    gr.update(choices=[], value=None))
        a_html = _preview_html(rel)
        # B dropdown: all OTHER assets
        b_choices = _all_other_asset_choices(exclude_rel=rel)
        b_choice_labels = [(fn, rp) for fn, rp in b_choices]
        return (gr.update(visible=True), a_html, gr.update(value='<em>Pick asset B from the dropdown above.</em>'),
                '<em>Pick asset B to see comparison stats.</em>',
                gr.update(choices=b_choice_labels, value=None))

    def on_compare_close():
        """Close the compare modal."""
        return gr.update(visible=False)

    def on_zip_export_selected(checked_paths):
        """Export the assets checked in the batch CheckboxGroup to a ZIP."""
        zip_path, status_html, file_update = _zip_export_assets(checked_paths)
        return status_html, file_update

    def on_zip_export_filtered():
        """Export all currently-filtered assets to a ZIP."""
        checked = [a[0] for a in _list_assets(
            tag_filter=gr_state.get('tag'),
            format_filter=gr_state.get('format'),
            favorite_only=gr_state.get('fav'),
            size_class_filter=gr_state.get('size_class'),
            grounded_only=gr_state.get('grounded'),
            search=gr_state.get('search', ''),
            untagged_only=gr_state.get('untagged', False),
            has_colliders_only=gr_state.get('has_colliders', False),
            recent_days=gr_state.get('recent_days', 0))
            if a[0] is not None]
        zip_path, status_html, file_update = _zip_export_assets(checked)
        return status_html, file_update

    def on_compare_b_select(b_rel):
        """Update B preview and stats when user picks B from the dropdown."""
        a_rel = gr_state.get('selected_path', '')
        if not b_rel or b_rel not in META['assets']:
            return (gr.update(value='<em>Pick a valid asset B.</em>'),
                    gr.update(value='<em>No B selected.</em>'))
        gr_state['compare_b'] = b_rel
        b_html = _preview_html(b_rel)
        # Build side-by-side stats
        stats = _compare_stats_html(a_rel, b_rel)
        return b_html, stats

    # Wire up - filter changes (now with 8 params)
    help_modal_close_btn.click(on_help_modal_close, None, help_modal)
    filter_inputs = [search_box, tag_dd, fmt_dd, size_dd, grounded_cb, fav_cb,
                    untagged_cb, has_colliders_cb, recent_sl]
    for comp in [search_box, tag_dd, fmt_dd, size_dd, grounded_cb, fav_cb,
                 untagged_cb, has_colliders_cb, recent_sl]:
        comp.change(on_search, filter_inputs, gallery)
    # Sort / group / fullscreen
    sort_dd.change(on_sort_change, [sort_dd, sort_asc_cb, group_dd], [gallery, count_md])
    sort_asc_cb.change(on_sort_change, [sort_dd, sort_asc_cb, group_dd], [gallery, count_md])
    group_dd.change(on_sort_change, [sort_dd, sort_asc_cb, group_dd], [gallery, count_md])
    fullscreen_btn.click(on_fullscreen, None, [fullscreen_col, fullscreen_html])
    fullscreen_close_btn.click(on_fullscreen_close, None, [fullscreen_col, fullscreen_html])
    help_btn.click(on_help_modal, None, help_modal)
    # Gallery / edit
    gallery.select(on_select, None, [sel_path, tags_box, notes_box, preview, siblings_dd])
    save_btn.click(on_save, [sel_path, tags_box, notes_box], [gallery, preview])
    siblings_dd.change(on_sibling_select, siblings_dd, [sel_path, tags_box, notes_box, preview, siblings_dd])
    fav_btn.click(on_fav, sel_path, [gallery, preview])
    cycle_tier_btn.click(on_cycle_tier, sel_path, [gallery, preview])
    compare_btn = gr.Button('Compare with another (D)', elem_id='compare-btn')
    compare_btn.click(
        on_compare,
        None,
        [compare_modal, compare_a_html, compare_b_html, compare_stats_html, compare_b_dd],
    )
    compare_close_btn.click(on_compare_close, None, compare_modal)
    compare_b_dd.change(on_compare_b_select, compare_b_dd, [compare_b_html, compare_stats_html])
    prev_btn.click(lambda: on_prev_next(-1), None, [sel_path, tags_box, notes_box, preview, siblings_dd])
    next_btn.click(lambda: on_prev_next(1), None, [sel_path, tags_box, notes_box, preview, siblings_dd])
    # Batch actions
    batch_tag_btn.click(
        lambda checked, tag: (_batch_apply_tag(checked, tag), []),
        [batch_assets_cg, batch_tag_tb],
        [batch_assets_cg, batch_assets_cg, batch_tag_tb],
    )
    batch_tier_btn.click(
        lambda checked, tier: _batch_set_tier(checked, tier),
        [batch_assets_cg, batch_tier_dd],
        [batch_assets_cg, batch_assets_cg],
    )
    batch_delete_btn.click(
        lambda checked: _batch_delete_from_meta(checked),
        [batch_assets_cg],
        [batch_assets_cg, batch_assets_cg],
    )
    batch_rename_btn.click(
        lambda checked, mode, text: _batch_rename(checked, mode, text),
        [batch_assets_cg, batch_rename_mode, batch_rename_tb],
        [batch_assets_cg, batch_assets_cg],
    )
    export_selected_btn.click(
        on_zip_export_selected,
        batch_assets_cg,
        [export_status_html, export_file],
    )
    export_filtered_btn.click(
        on_zip_export_filtered,
        None,
        [export_status_html, export_file],
    )
    preset_save_btn.click(
        on_save_preset,
        [preset_name_tb, search_box, tag_dd, fmt_dd, size_dd, grounded_cb, fav_cb, untagged_cb, has_colliders_cb, recent_sl],
        preset_save_btn,
    )
    preset_load_btn.click(
        on_load_preset,
        preset_dd,
        [search_box, tag_dd, fmt_dd, size_dd, grounded_cb, fav_cb, untagged_cb, has_colliders_cb, recent_sl, gallery, count_md, preset_dd, preset_dd],
    )
    preset_delete_btn.click(on_delete_preset, preset_dd, [preset_dd, preset_dd])

clear_output()
def _on_welcome():
    return ('<div style="background: #e3f2fd; padding: 16px; border-radius: 8px;">'
            '<b>Welcome to the Asset Library Browser!</b><br>'
            '<small>Pick an asset from the gallery, or use the <b>Filter</b> sidebar on the left to narrow by tag, format, size class, etc. '
            'Press <kbd>?</kbd> for keyboard shortcuts, <kbd>F</kbd> to focus search, <kbd>D</kbd> to compare two assets, '
            '<kbd>T</kbd> to cycle the tier on the selected asset.</small></div>')
demo.load(_on_welcome, None, preview)
demo.queue(max_size=4, default_concurrency_limit=2)
demo.launch(share=False, show_error=True, height=900, quiet=False)


In [ ]:
# @title STEP 4 — Batch Render Thumbnails (optional)
# @markdown Renders 256×256 PNG thumbnails for every MESH asset. The Gradio
# @markdown gallery shows these alongside filenames. 3DGS assets are skipped
# @markdown (no easy in-notebook renderer; the Gradio UI shows a file icon).
# @markdown Run once after the library grows; idempotent (only renders missing).

import os, sys, json, time, pathlib
import numpy as np
from PIL import Image
import trimesh
from tqdm import tqdm

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
THUMB_DIR = LIBRARY_DIR / '_thumbs'
THUMB_DIR.mkdir(exist_ok=True)
with open(META_FILE) as f:
    META = json.load(f)

THUMB_SIZE = 256  # @param {type:"slider", min:64, max:1024, step:64}
# @markdown *Thumbnail edge size in pixels. 256 = gallery-friendly. 512 = preview-quality.*
RENDER_RES = 256  # @param {type:"slider", min:128, max:2048, step:64}
# @markdown *Mesh render resolution. Higher = better quality, slower.*
OVERWRITE = False  # @param {type:"boolean"}
# @markdown *Re-render existing thumbnails. Default False (only render missing).*
MESH_FORMATS = ('MESH',)  # only render mesh files; skip 3DGS / images / text

def _render_mesh_thumb(mesh_path, out_path, size=256, res=256):
    """Render a trimesh mesh to a PNG thumbnail via offscreen pyrender.
    Falls back to a placeholder PNG if pyrender/OpenGL is unavailable."""
    try:
        m = trimesh.load(str(mesh_path), force='mesh')
        if not isinstance(m, trimesh.Trimesh) or len(m.vertices) == 0:
            return False
        # Center + normalize to unit cube for consistent thumbnail framing
        m.vertices -= (m.vertices.min(axis=0) + m.vertices.max(axis=0)) / 2.0
        extent = float((m.vertices.max(axis=0) - m.vertices.min(axis=0)).max())
        if extent > 1e-9:
            m.vertices /= extent
        # Use trimesh's built-in offscreen renderer
        try:
            scene = m.scene()
            png = scene.save_image(resolution=(res, res), background=[20, 20, 30, 255])
            if png:
                with open(out_path, 'wb') as f:
                    f.write(png)
                return True
        except Exception as e:
            pass
        # Fallback: try pyglet
        try:
            from trimesh.viewer import SceneViewer
            return False  # not a real fallback, just signal failure
        except Exception:
            pass
        return False
    except Exception as e:
        print(f'[thumb] render failed for {mesh_path}: {e}')
        return False

def _placeholder_thumb(out_path, size=256, label=''):
    """Generate a placeholder PNG with the file extension as a label."""
    img = Image.new('RGB', (size, size), (40, 40, 50))
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 32)
        small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
    except Exception:
        font = ImageFont.load_default()
        small = font
    if label:
        # Center the label
        bbox = draw.textbbox((0, 0), label, font=font)
        w = bbox[2] - bbox[0]
        draw.text(((size - w) / 2, size / 2 - 20), label, fill=(220, 220, 230), font=font)
    img.save(out_path)

t0 = time.time()
n_total = n_rendered = n_skipped = n_failed = 0
for rel, m in tqdm(META['assets'].items(), desc='thumbnails'):
    if m['format'] not in MESH_FORMATS:
        n_skipped += 1
        continue
    n_total += 1
    thumb_path = THUMB_DIR / (rel.replace('/', '__') + '.png')
    if thumb_path.exists() and not OVERWRITE:
        continue
    src_path = LIBRARY_DIR / rel
    if _render_mesh_thumb(src_path, thumb_path, size=THUMB_SIZE, res=RENDER_RES):
        n_rendered += 1
    else:
        # Placeholder with the extension as label (e.g. "GLB")
        ext = src_path.suffix.lstrip('.').upper()
        _placeholder_thumb(thumb_path, size=THUMB_SIZE, label=ext)
        n_failed += 1
print(f'\n[thumb] Done in {time.time()-t0:.1f}s')
print(f'  MESH assets: {n_total}, rendered: {n_rendered}, failed (placeholder): {n_failed}, skipped (non-mesh): {n_skipped}')
print(f'\n[Done] Thumbnails in {THUMB_DIR}/. Refresh the Gradio UI in STEP 3 to see them.')


In [ ]:
# @title STEP 5 — Export: Unity / Godot / Static HTML Portfolio / CSV
# @markdown Exports the library in formats game engines and portfolio sites
# @markdown can ingest. Each export respects the current filter (tag / format /
# @markdown favorite / search) so you can ship a curated subset.

import os, json, time, shutil, zipfile, pathlib
import pandas as pd
from datetime import datetime, timezone

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
THUMB_DIR = LIBRARY_DIR / '_thumbs'
EXPORT_DIR = LIBRARY_DIR / '_exports'
EXPORT_DIR.mkdir(exist_ok=True)
with open(META_FILE) as f:
    META = json.load(f)

EXPORT_FORMATS = ['unity', 'godot', 'html_portfolio', 'csv_manifest']  # @param ["unity", "godot", "html_portfolio", "csv_manifest"]
# @markdown *Comma-separated list. Each export goes to /content/drive/MyDrive/AEI_3D_Out/_exports/<name>/.*
INCLUDED_FORMATS = ['MESH']  # @param ["MESH", "3DGS_RAW", "3DGS_PACKED", "IMAGE", "ALL"]
# @markdown *Filter to specific asset formats. MESH only (skip 3DGS). Use ALL to include everything.*
INCLUDED_TAGS = ''  # @param {type:"string"}
# @markdown *Comma-separated tag whitelist. Empty = no filter (include all). Example: 'hero, vehicle'.*
FAVORITES_ONLY = False  # @param {type:"boolean"}
# @markdown *Only export favorited assets.*

def _filter_assets():
    """Return the asset paths matching INCLUDED_FORMATS, INCLUDED_TAGS, FAVORITES_ONLY."""
    fmt_set = set(INCLUDED_FORMATS) if INCLUDED_FORMATS != ['ALL'] else None
    if INCLUDED_TAGS.strip():
        tag_set = set(t.strip() for t in INCLUDED_TAGS.split(',') if t.strip())
    else:
        tag_set = None
    out = []
    for rel, m in META['assets'].items():
        if fmt_set and m['format'] not in fmt_set:
            continue
        if tag_set and not (set(m.get('tags', [])) & tag_set):
            continue
        if FAVORITES_ONLY and not m.get('favorite', False):
            continue
        out.append((rel, m))
    return sorted(out)

assets = _filter_assets()
print(f'[export] Filtered: {len(assets)} assets match criteria')
if not assets:
    print('[export] WARNING: no assets match. Loosen filters.')
    print('[export]   Try: INCLUDED_FORMATS=[MESH,3DGS_RAW], FAVORITES_ONLY=False')
print()

ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

if 'csv_manifest' in EXPORT_FORMATS:
    print('[export] CSV manifest ...')
    csv_path = EXPORT_DIR / f'manifest_{ts}.csv'
    rows = []
    for rel, m in assets:
        rows.append({
            'path': rel,
            'filename': m.get('filename'),
            'format': m.get('format'),
            'size_mb': round(m.get('size_bytes', 0) / 1024 / 1024, 3),
            'tags': ', '.join(m.get('tags', [])),
            'favorite': m.get('favorite', False),
            'notes': m.get('notes', ''),
            'added': m.get('added', ''),
            'mtime': datetime.fromtimestamp(m.get('mtime', 0), timezone.utc).isoformat() if m.get('mtime') else '',
        })
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f'  -> {csv_path} ({len(rows)} rows)')

if 'unity' in EXPORT_FORMATS:
    print('[export] Unity AssetBundle-like folder structure ...')
    unity_dir = EXPORT_DIR / f'unity_{ts}'
    unity_dir.mkdir(exist_ok=True)
    (unity_dir / 'Assets').mkdir(exist_ok=True)
    (unity_dir / 'Assets' / 'AEI_Library').mkdir(exist_ok=True)
    (unity_dir / 'Assets' / 'AEI_Library' / 'Meshes').mkdir(exist_ok=True)
    (unity_dir / 'Assets' / 'AEI_Library' / 'Thumbs').mkdir(exist_ok=True)
    n = 0
    for rel, m in assets:
        if m['format'] != 'MESH':
            continue
        src = LIBRARY_DIR / rel
        dst = unity_dir / 'Assets' / 'AEI_Library' / 'Meshes' / m['filename']
        if not dst.exists():
            shutil.copy2(src, dst)
        thumb = THUMB_DIR / (rel.replace('/', '__') + '.png')
        if thumb.exists():
            shutil.copy2(thumb, unity_dir / 'Assets' / 'AEI_Library' / 'Thumbs' / thumb.name)
        n += 1
    # README with import instructions
    readme = unity_dir / 'Assets' / 'AEI_Library' / 'README.md'
    readme.write_text(f'''# AEI Asset Library \u2192 Unity

**Generated:** {datetime.now(timezone.utc).isoformat()}
**Source:** {LIBRARY_DIR}
**Asset count:** {n} meshes (Unity prefers .fbx for rigged assets, but .glb and .obj import fine for static props)

## Import into Unity

1. Copy the `Assets/AEI_Library` folder into your Unity project's `Assets/`.
2. Unity auto-imports .glb / .obj / .fbx. The default scale is 1 unit = 1 meter
   (matches our game-asset transform defaults).
3. For the thumbs folder: select any mesh in the Project window and the
   thumbnail preview will appear.
4. To bundle into an AssetBundle: select all meshes in `AEI_Library/Meshes/`,
   set their AssetBundle name in the Inspector (e.g. `aei/library`),
   then build via `BuildPipeline.BuildAssetBundles`.

## Notes

- These meshes are untextured. Texture them in your DCC tool of choice
  (Blender, Substance Painter) before shipping.
- Use Unity's mesh decimation (in the Model import settings, Model tab)
  to make per-platform LODs.
''')
    print(f'  -> {unity_dir}/ (with README, {n} meshes copied)')

if 'godot' in EXPORT_FORMATS:
    print('[export] Godot mesh library (file copy + .import sidecar) ...')
    godot_dir = EXPORT_DIR / ('godot_' + ts)
    godot_dir.mkdir(exist_ok=True)
    n = 0
    for rel, m in assets:
        if m['format'] != 'MESH':
            continue
        src = LIBRARY_DIR / rel
        # Godot auto-detects .glb/.obj/.fbx/.stl on import.
        # Just copy the file; Godot will generate the .import sidecar on first scan.
        target = godot_dir / m['filename']
        if not target.exists():
            shutil.copy2(src, target)
        n += 1
    # README with import instructions
    readme = godot_dir / 'README.md'
    readme_lines = [
        '# AEI Asset Library to Godot',
        '',
        'Drop this entire folder into your Godot project root, then in the Godot editor:',
        '- FileSystem panel: right-click to Reimport',
        '- For .glb/.gltf: enable the import-on-drop option',
        '- For .fbx: install the FBX importer plugin',
        '',
        'Meshes are untextured. Texture them in Blender before shipping.',
    ]
    readme.write_text('\n'.join(readme_lines))
    print('  -> {godot_dir}/ ({n} mesh files + README)'.format(godot_dir=str(godot_dir), n=n))
    print('  Drop the folder into your Godot project root.')


print('\n[Done] Exports in /content/drive/MyDrive/AEI_3D_Out/_exports/.')


In [ ]:
# @title STEP 6 — Stats Dashboard
# @markdown A textual summary of your library: format breakdown, top tags,
# @markdown biggest files, missing/broken files report. Useful for QA before
# @markdown shipping a 200+ asset pack.

import os, json, pathlib
from collections import Counter
import pandas as pd

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
with open(META_FILE) as f:
    META = json.load(f)
assets = META['assets']
n = len(assets)

print('=== AEI Asset Library Stats ===\n')
print(f'Library dir: {LIBRARY_DIR}')
print(f'Total assets: {n}')
total_size = sum(m.get('size_bytes', 0) for m in assets.values())
print(f'Total size:   {total_size/1024/1024:.1f} MB ({total_size/1024**3:.2f} GB)\n')

# Format breakdown
print('--- Format breakdown ---')
fmt_counts = Counter(m['format'] for m in assets.values())
fmt_sizes = Counter()
for m in assets.values():
    fmt_sizes[m['format']] += m.get('size_bytes', 0)
for fmt, n_f in fmt_counts.most_common():
    sz_mb = fmt_sizes[fmt] / 1024 / 1024
    print(f'  {fmt:15s}: {n_f:4d} files, {sz_mb:7.1f} MB')

# Size class breakdown (from <slug>_meta.json sidecars)
sc_counts = Counter(m.get('size_class') or '(none)' for m in assets.values())
if any(k != "(none)" for k in sc_counts):
    for sc, n_s in sc_counts.most_common():
        if sc == '(none)':
            print(f'  {sc:18s}: {n_s:4d} files (no meta.json sidecar)')
        else:
            print(f'  {sc:18s}: {n_s:4d} files')
else:
    print('  (no size_class data - run TripoSPlat STEP 8 first to generate meta.json sidecars)')

# Grounding status
n_grounded = sum(1 for m in assets.values()
                  if (m.get('grounding') or {}).get('applied'))
n_with_grounding_meta = sum(1 for m in assets.values()
                                if m.get('grounding') is not None)
n_total = n
print(f'\nGrounded: {n_grounded} of {n_with_grounding_meta} assets with grounding metadata ({100*n_grounded/max(1,n_with_grounding_meta):.1f}%)')
if n_with_grounding_meta < n_total:
    n_no_meta = n_total - n_with_grounding_meta
    print(f'  (no grounding data: {n_no_meta} assets - these are not from TripoSPlat STEP 8)')


# Top tags
print('\n--- Top 15 tags ---')
tag_counts = Counter()
for m in assets.values():
    for t in m.get('tags', []):
        tag_counts[t] += 1
for tag, n_t in tag_counts.most_common(15):
    print(f'  {tag:30s}: {n_t:4d} assets')

# Favorites
n_fav = sum(1 for m in assets.values() if m.get('favorite', False))
print(f'\nFavorited: {n_fav} ({100*n_fav/max(1,n):.1f}%)')

# Biggest files
print('\n--- Top 10 biggest files ---')
biggest = sorted(assets.values(), key=lambda m: m.get('size_bytes', 0), reverse=True)[:10]
for m in biggest:
    sz = m.get('size_bytes', 0) / 1024 / 1024
    print(f'  {sz:7.1f} MB  {m["format"]:12s}  {m["path"]}')

# Missing/broken files
print('\n--- Missing files (in metadata but not on disk) ---')
missing = []
for rel, m in assets.items():
    p = LIBRARY_DIR / rel
    if not p.exists():
        missing.append(rel)
if missing:
    for rel in missing[:20]:
        print(f'  MISSING: {rel}')
    if len(missing) > 20:
        print(f'  ... and {len(missing) - 20} more')
else:
    print('  (none \u2014 all metadata entries have files on disk)')

# Orphaned files (on disk but not in metadata)
print('\n--- Orphaned files (on disk but not in metadata) ---')
all_disk = set()
for p in LIBRARY_DIR.rglob('*'):
    if p.is_file() and not p.name.startswith('.') and '_thumbs' not in p.parts and '_exports' not in p.parts and p.name != '_library_meta.json':
        all_disk.add(str(p.relative_to(LIBRARY_DIR)))
all_meta = set(assets.keys())
orphans = all_disk - all_meta
if orphans:
    for rel in sorted(orphans)[:20]:
        print(f'  ORPHAN: {rel}')
    if len(orphans) > 20:
        print(f'  ... and {len(orphans) - 20} more')
else:
    print('  (none)')
print('\n[Done] Run STEP 2 again to refresh metadata if you see MISSING/ORPHAN.')


In [ ]:
# @title STEP 7 — Keep-Alive (Gradio stays alive)
# @markdown Standard pattern. Gradio will keep running as long as this kernel is
# @markdown alive. The KEEP_ALIVE_JS prevents Colab from idling you out.

import IPython
from google.colab import output
KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Started \u2014 pings every 30 min.')


In [ ]:
# @title STEP 8 — Help / Format Reference / Known Issues
# @markdown Reference material. Read this if anything in the browser went wrong.

help_md = '''
# Asset Library Browser \u2014 Help, Format Reference, and Known Issues

## What this notebook is for

After running TripoSplat + GauStudio / SuGaR + Mesh Optimizer on your 200+ images, you have a folder full of assets but no way to:
- Browse them all at once
- Preview 3D meshes in your browser
- Tag / favorite the good ones
- Export a curated subset to Unity / Godot / a portfolio site

This notebook is the missing piece.

## Format reference

| Format | Category | Preview support |
|--------|----------|------------------|
`.ply` (3DGS-extended) | `3DGS_RAW` | Metadata card + link to external 3DGS viewer |
`.splat` (packed) | `3DGS_PACKED` | Metadata card + link to Antimatter15 viewer |
`.glb` (binary glTF) | `MESH` | Inline `<model-viewer>` (best UX) |
`.obj` (Wavefront) | `MESH` | Inline `<model-viewer>` |
`.fbx` (FBX 7.4) | `MESH` | Inline `<model-viewer>` |
`.ply` (mesh) | `MESH` | Inline `<model-viewer>` |
`.stl` (3D print) | `MESH` | Inline `<model-viewer>` |
`.3mf` (3D Manuf.) | `MESH` | Inline `<model-viewer>` |
`.png` / `.jpg` / `.webp` | `IMAGE` | Inline `<img>` |
`.txt` / `.json` / `.zip` | `TEXT` / `META` / `ARCHIVE` | File info only |

## 3DGS preview workaround

`<model-viewer>` doesn't support 3DGS Gaussian Splats. For 3DGS files you get a metadata card with a link to Antimatter15's online viewer (or LumaAI, gsplat.tech). To get inline previews, run GauStudio on the 3DGS PLY \u2192 get a GLB mesh \u2192 come back to the browser and preview the GLB.

## Thumbnails

STEP 4 renders 256\u00d7256 PNG thumbnails via trimesh's offscreen renderer for every MESH asset. 3DGS files get a placeholder with the extension as a label. Re-run after adding new assets.

## Exports

| Export | What's in it | Game-engine import |
|--------|--------------|-------------------|
Unity | `Assets/AEI_Library/Meshes/` + thumbnails + README | Copy `Assets/AEI_Library/` into your Unity project's `Assets/` |
Godot | `*.tres` mesh resources + actual mesh files | Copy into your Godot project's root, then use `FileSystem > Import` |
HTML portfolio | Self-contained `index.html` + thumbs + meshes | Upload to GitHub Pages, Netlify, or any static host |
CSV manifest | `manifest_<ts>.csv` with path, format, tags, size, etc. | Open in Excel / Google Sheets for inventory |

## Known issues

1. **3DGS files get no thumbnail**: no easy in-notebook 3DGS renderer. Workaround: run GauStudio on them first, then re-scan.
2. **Large meshes (>50 MB) slow to render thumbnails**: trimesh's offscreen renderer loads the whole mesh in memory. For 200 assets, expect 2-5 min total render time.
3. **`<model-viewer>` doesn't show texture for GLB if it references external files**: our GLBs are self-contained (texture embedded) so this shouldn't happen. If you see a black mesh, check that the GLB has a `.bin` or embedded texture.
4. **Gradio URL only works while the kernel is alive**: when the session ends, the Gradio URL dies. To share, export the HTML portfolio (STEP 5) and host it on a static site.
5. **Drive sync conflicts**: if multiple notebooks are writing to the same `_library_meta.json`, you can lose tag changes. STEP 2 reads the sidecar on each run; STEP 3's save button writes immediately. Avoid running two browsers in parallel.

## SplatTransform output formats (from [SplatTransform_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SplatTransform_Colab.ipynb))

The PlayCanvas `splat-transform` post-processor takes raw 3DGS `.ply` files and converts them to compressed game-engine formats. If you've run SplatTransform_Colab on the same library, you'll see those files here too. Key formats to recognize:

- **`.sog`** (PlayCanvas native, ~10x compression) — open in [playcanvas.com/viewer](https://playcanvas.com/viewer/)
- **`.spz`** (Niantic, smallest, mobile-friendly) — Scaniverse
- **`.ksplat`** (mkkellogg) — community 3DGS viewers
- **`.lcc`** (XGRIDS) — LumaAI native
- **`.glb` with `KHR_gaussian_splatting`** (the new glTF 2.0 standard extension) — auto-detected by header magic. Future-proof for Three.js / Babylon.js / Unity / Unreal.
- **`-collision.glb`** (voxel marching-cubes mesh, runtime physics) — auto-detected by filename suffix. NOT a textured visual mesh.
- **`lod-meta.json`** (PlayCanvas streamed SOG metadata) — auto-detected by exact filename. Pair with the .sog files in the same folder for progressive loading.

The browser recognizes all of these (via the format classification in STEP 2) and shows a metadata card with the right viewer link for each. Run STEP 4 to render thumbnails for the mesh files; 3DGS files get no thumbnail (no easy in-notebook renderer) but are still scannable + filterable.

## See also

- **TripoSplat_Colab.ipynb** \u2014 generate 3DGS + mesh from a single image
- **SuGaR_Colab.ipynb** \u2014 high-quality mesh from 3DGS PLY (sharp, slow)
- **GauStudio_Colab.ipynb** \u2014 fast mesh from 3DGS PLY (smooth, fast)
- **Mesh_Optimizer_Colab.ipynb** \u2014 post-process meshes (UV unwrap, fill holes, etc.)
- **TripoSplat STEP 7** \u2014 batch generation (use this to build the 200+ library)
- **TripoSplat STEP 8** \u2014 post-process existing meshes (use to convert 3DGS to mesh for previews)
'''
from IPython.display import Markdown, display
display(Markdown(help_md))
print(help_md)
